# ABS Quarterly Construction Work Done 8755

Value of construction work done: building and engineering, by sector, state and
type of work. Complements the dwelling counts in 8752 (Building Activity) with the
dollar value of all construction, including non-residential building and
engineering.

## Python set-up

In [1]:
# analytic imports
import pandas as pd
import readabs as ra
from readabs import metacol as mc
from mgplot import (
    get_color,
    line_plot_finalise,
    multi_start,
    seastrend_plot_finalise,
    series_growth_plot_finalise,
)

# local imports
from abs_gdp import get_gdp
from abs_helper import get_abs_data

# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 999

In [2]:
# display and file settings
SHOW = False
FILE_TYPE = "png"

# chart time ranges
plot_times = 0, -20  # full history, last 5 years (quarterly)
GROWTH_PLOT_FROM = -19  # growth charts: last ~5 years of quarters

ANNUAL_QTRS = 4  # quarters in a year: annualising window
PERCENT = 100
THOUSANDS_PER_MILLION = 1_000
INDEX_BASE = pd.Period("2019Q4", freq="Q-DEC")  # pre-COVID base for indexing

# 8755.0 table names
CVM_TABLE = "8755001"  # by sector, chain volume measures
CP_TABLE = "8755002"  # by sector, current prices
BUILDING_CP_TABLE = "8755006"  # building by type of work, current prices
STATE_CVM_TABLE = "8755008"  # by state, chain volume measures
ENG_TYPE_TABLE = "87550010"  # engineering by type, current prices, Original only

# data item description fragments -> chart labels
CONSTRUCTION_TYPES = {
    "Total building": "Building",
    "Engineering Construction": "Engineering",
    "Total (Type of Construction)": "All Construction",
}
SECTORS = ("Total Sectors", "Private Sector", "Public Sector")
SEAS_TREND = ("Seasonally Adjusted", "Trend")
SA = "Seasonally Adjusted"

# current prices do not split new residential into houses and other residential
BUILDING_WORK_TYPES = {
    "New ;  Total Residential ;": "New residential",
    "Alterations and additions including conversions ;  Total Residential ;": "Alterations and additions",
    "Total (Type of Work) ;  Total Non-residential ;": "Non-residential",
}
ENG_WORK_TYPES = {
    "Roads, highways and subdivisions": "Roads",
    "Railways": "Railways",
    "Electricity generation, transmission and distribution": "Electricity",
    "Oil, gas, coal and other minerals": "Oil, gas, coal and minerals",
    "Water storage and supply": "Water",
    "Telecommunications": "Telecommunications",
}
COMPARE_STATES = ("New South Wales", "Victoria", "Queensland", "Western Australia")

## Get data from ABS

In [3]:
abs_dict, meta, source, RECENT = get_abs_data("8755.0")

Table 87550do001_202606 has no 'Index' sheet.
Table 87550do002_202606 has no 'Index' sheet.
Table 87550do003_202606 has no 'Index' sheet.
Table 87550do004_202606 has no 'Index' sheet.
Table 87550do005_202606 has no 'Index' sheet.


In [4]:
def get_series(table: str, did: str, stype: str) -> tuple[pd.Series, str]:
    """Select one 8755 series by table, data item description and series type.

    Raises if the selected series is empty, so a renamed or discontinued series
    fails here rather than as a blank chart.
    """
    _t, sid, units = ra.find_abs_id(
        meta, {table: mc.table, did: mc.did, stype: mc.stype}, verbose=False
    )
    series = abs_dict[table][sid].dropna()
    if series.empty:
        raise ValueError(f"No data for {stype} {did!r} in table {table}")
    return series, units

In [5]:
def get_nominal_gdp(seasonal: str) -> pd.Series:
    """Nominal GDP (5206.0) in $ Millions; seasonal is SA, T or O."""
    gdp, gdp_units = get_gdp(gdp_type="CP", seasonal=seasonal)
    if gdp_units != "$ Millions":
        raise ValueError(f"Expected nominal GDP in $ Millions, got {gdp_units!r}")
    return gdp


def as_share_of_gdp(work: pd.Series, units: str, gdp: pd.Series) -> pd.Series:
    """Express current-price work done ($'000) as a percentage of nominal GDP ($ Millions).

    The two series must be the same kind of flow (both quarterly, or both annualised).
    """
    if units != "$'000":
        raise ValueError(f"Expected work done in $'000, got {units!r}")
    return (work / THOUSANDS_PER_MILLION / gdp * PERCENT).dropna()

## Plot

### Headline: work done by sector and type of construction

In [6]:
def plot_headline() -> None:
    """Seasonally adjusted and trend work done (CVM), by sector and construction type."""

    for sector in SECTORS:
        for ctype, label in CONSTRUCTION_TYPES.items():
            did = f"Value of work done ;  Chain Volume Measures ;  {sector} ;  {ctype} ;"
            parts = {}
            units = ""
            for stype in SEAS_TREND:
                parts[stype], units = get_series(CVM_TABLE, did, stype)
            frame, units = ra.recalibrate(pd.DataFrame(parts), units)
            multi_start(
                frame,
                function=seastrend_plot_finalise,
                starts=plot_times,
                title=f"Work Done: {label}: {sector}",
                ylabel=f"{units}/Quarter",
                lfooter="Australia. Chain volume measures. ",
                rfooter=source,
                show=SHOW,
                file_type=FILE_TYPE,
            )


plot_headline()

### Growth

In [7]:
def plot_growth() -> None:
    """Quarterly and through-the-year growth in work done (CVM, SA, all sectors)."""

    for ctype, label in CONSTRUCTION_TYPES.items():
        did = f"Value of work done ;  Chain Volume Measures ;  Total Sectors ;  {ctype} ;"
        series, _units = get_series(CVM_TABLE, did, SA)
        series_growth_plot_finalise(
            series,
            plot_from=GROWTH_PLOT_FROM,
            title=f"Growth in Work Done: {label}",
            lfooter="Australia. Seasonally adjusted. Chain volume measures. ",
            rfooter=source,
            show=SHOW,
            file_type=FILE_TYPE,
        )


plot_growth()

### Composition: building v engineering, private v public

In [8]:
def plot_composition() -> None:
    """Building v engineering (all sectors), and private v public (all construction)."""

    comparisons = {
        "Work Done: Building v Engineering": {
            label: f"Total Sectors ;  {ctype} ;"
            for ctype, label in CONSTRUCTION_TYPES.items()
            if ctype != "Total (Type of Construction)"
        },
        "Work Done: Private v Public Sector": {
            sector: f"{sector} ;  Total (Type of Construction) ;"
            for sector in SECTORS
            if sector != "Total Sectors"
        },
    }
    for title, wanted in comparisons.items():
        parts = {}
        units = ""
        for label, fragment in wanted.items():
            did = f"Value of work done ;  Chain Volume Measures ;  {fragment}"
            parts[label], units = get_series(CVM_TABLE, did, SA)
        frame, units = ra.recalibrate(pd.DataFrame(parts), units)
        multi_start(
            frame,
            function=line_plot_finalise,
            starts=plot_times,
            title=title,
            ylabel=f"{units}/Quarter",
            annotate=True,
            legend={"loc": "best", "fontsize": "x-small"},
            lfooter="Australia. Seasonally adjusted. Chain volume measures. ",
            rfooter=source,
            show=SHOW,
            file_type=FILE_TYPE,
        )


plot_composition()

### Work done as a share of GDP

Both sides are quarterly flows in current prices, seasonally adjusted.

In [9]:
def plot_share_of_gdp() -> None:
    """Work done (current prices, SA) as a percentage of nominal GDP (SA)."""

    gdp = get_nominal_gdp("SA")
    shares = {}
    for ctype, label in CONSTRUCTION_TYPES.items():
        did = f"Value of work done ;  Current Prices ;  Total Sectors ;  {ctype} ;"
        work, units = get_series(CP_TABLE, did, SA)
        shares[label] = as_share_of_gdp(work, units, gdp)
    frame = pd.DataFrame(shares).dropna()

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=plot_times,
        title="Construction Work Done as a Share of GDP",
        ylabel="Per cent of GDP",
        annotate=True,
        legend={"loc": "best", "fontsize": "x-small"},
        lfooter="Australia. Seasonally adjusted. Current prices. ",
        rfooter="ABS: 8755.0 5206.0",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_share_of_gdp()

### Building work by type of work

In [10]:
def plot_building_by_work_type() -> None:
    """Building work done by type of work (current prices, SA) as a share of nominal GDP."""

    gdp = get_nominal_gdp("SA")
    parts = {}
    for fragment, label in BUILDING_WORK_TYPES.items():
        did = (
            "Value of work done during quarter ;  Current Prices ;  "
            f"Total Sectors ;  {fragment}"
        )
        work, units = get_series(BUILDING_CP_TABLE, did, SA)
        parts[label] = as_share_of_gdp(work, units, gdp)
    frame = pd.DataFrame(parts).dropna()

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=plot_times,
        title="Building Work Done by Type of Work as a Share of GDP",
        ylabel="Per cent of GDP",
        annotate=True,
        legend={"loc": "best", "fontsize": "x-small"},
        lfooter="Australia. Seasonally adjusted. Current prices. All sectors. ",
        rfooter="ABS: 8755.0 5206.0",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_building_by_work_type()

### Engineering work by type of construction

Published in current prices and unadjusted only, so annualised with a rolling
4-quarter sum to remove the seasonal pattern.

In [11]:
def plot_engineering_by_type() -> None:
    """Engineering work done by type (all sectors) as a share of nominal GDP.

    Both sides are Original current-price series annualised with a rolling
    4-quarter sum, which removes the seasonal pattern.
    """

    annual_gdp = get_nominal_gdp("O").rolling(ANNUAL_QTRS).sum()
    parts = {}
    for eng_type, label in ENG_WORK_TYPES.items():
        did = f"Value of work done ;  Total all sectors ;  {eng_type} ;"
        work, units = get_series(ENG_TYPE_TABLE, did, "Original")
        parts[label] = as_share_of_gdp(work.rolling(ANNUAL_QTRS).sum(), units, annual_gdp)
    frame = pd.DataFrame(parts).dropna()

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=plot_times,
        title="Engineering Work Done by Type of Construction as a Share of GDP",
        ylabel="Per cent of GDP",
        annotate=True,
        legend={"loc": "best", "fontsize": "x-small"},
        lfooter="Australia. Original series. Current prices. All sectors. "
        "Annualised (rolling 4-quarter sums). ",
        rfooter="ABS: 8755.0 5206.0",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_engineering_by_type()

### States

Indexed to a pre-COVID base, so states of different size compare proportionally.

In [12]:
def plot_states() -> None:
    """Total construction work done (CVM, SA) by state, indexed to INDEX_BASE."""

    for ctype, label in CONSTRUCTION_TYPES.items():
        parts = {}
        for state in COMPARE_STATES:
            did = f"Value of work done ;  Chain Volume Measures ;  {state} ;  {ctype} ;"
            series, _units = get_series(STATE_CVM_TABLE, did, SA)
            parts[state] = series / series[INDEX_BASE] * PERCENT
        frame = pd.DataFrame(parts)

        multi_start(
            frame,
            function=line_plot_finalise,
            starts=plot_times,
            title=f"Work Done by State: {label}",
            ylabel=f"Index ({INDEX_BASE} = 100)",
            color=[get_color(s) for s in COMPARE_STATES],
            annotate=True,
            legend={"loc": "best", "fontsize": "x-small"},
            lfooter="Australia. Seasonally adjusted. Chain volume measures. ",
            rfooter=source,
            show=SHOW,
            file_type=FILE_TYPE,
        )


plot_states()

## Finished

In [13]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-09-28 10:48:24

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.17.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 27.0.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.3.0
pandas : 3.0.6
readabs: 0.3.0

Watermark: 2.6.0



In [14]:
print("Finished")

Finished
